# 05 - Behavioural Novelty + Risk Engine

`novelty_score` answers a different question than the classifier: *"how
unusual is this behaviour compared with legitimate traffic?"*. It is the ECDF
rank of an anomaly model (fit on legitimate training rows) relative to
legitimate calibration traffic, so 0.90 means *more atypical than 90% of
legitimate transactions*.


In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
import numpy as np, pandas as pd, os, joblib
from src.config import get_settings
from src.feature_engineering import NOVELTY_FEATURES

cfg = get_settings()
novelty = joblib.load(os.path.join(cfg.models_dir(), "novelty.joblib"))
print("novelty model:", novelty.name, "| ECDF ref n:", novelty.ref_n)


In [ ]:
tr = pd.read_parquet(os.path.join(cfg.processed_dir(), "train_features.parquet"))
va = pd.read_parquet(os.path.join(cfg.processed_dir(), "val_features.parquet"))
nov = novelty.novelty(va[NOVELTY_FEATURES])
s = pd.Series(nov)
print("novelty distribution (val):")
print(s.describe().round(3).to_string())
print("share > 0.90 (top decile of attention):",
      round(float((s > 0.90).mean()), 4))


In [ ]:
from src.risk_engine import RiskEngine, fit_thresholds
import json
thr = json.load(open(os.path.join(cfg.models_dir(), "thresholds.json")))
risk = RiskEngine(thr)
print("thresholds:", thr)

probe = [risk.decide(p, n) for p, n in [(0.001, 0.95), (0.05, 0.95),
                                        (0.55, 0.30), (0.80, 0.99)]]
for d in probe:
    print(f"  p={d['fraud_probability']:.3f} nov={d['novelty_score']:.2f} "
          f"-> {d['risk_band']:9s} alert={d['alert']}")


In [ ]:
from src.streaming_processor import AlertBudgetController
budget = AlertBudgetController(max_per_hour=250, max_per_day=3000,
                               budget_pct=2.0, high_risk_share_cap_pct=0.7)
status = [budget.decide(1.7e9 + i*3600, "review", 0.9 - i*0.02)
          for i in range(5)]
print("budget decisions:", status)
print("escalated:", budget.total_escalated, "logged:", budget.logged)


## Risk decision matrix
|                      | novelty low | novelty high |
|----------------------|-------------|--------------|
| **p low**            | normal      | monitor      |
| **p high**           | review      | high-risk    |

`high-risk` -> always alert; `review` -> alert; `monitor` -> logged, no
escalation; `normal` -> allow. A high novelty score **never** triggers an
alert on its own (new-but-normal customers are protected) – the novelty->
review channel requires `p >= p_floor`.
